# 1. Idealista: Municipios por Provincia
**Propósito**: Scrapea la página de una provincia en Idealista y extrae las URLs de todos sus municipios, extrae además el número de anuncios de cada municipio así como las páginas que hay que escrapear en cada caso (Idealista muestra 30 anuncios por página).

In [1]:
import sys
sys.path.insert(0, '/tfm/python_notebooks')
from pyspark.sql import functions as F
from datetime import datetime
from tfm_lib import idealista_scrap as idealista
from tfm_lib import utils as tfm_utils

In [2]:
debug_mode = False # True#: entorno TEST gratis (1 registro)
provincia = 'murcia'

#Construimos la url de donde escrapear los municipios de la provincia
provincia_url = f'https://www.idealista.com/venta-viviendas/{provincia}-provincia/municipios'

## Scraping de Municipios
Scrapeamos con Scrapfly la página de la provincia para recuperar la información de los municipios de dicha provincia.

In [3]:
municipios_urls = idealista.scraper.run_async(idealista.scraper.scrape_provinces([provincia_url], debug=debug_mode))

print(f'Municipios encontrados en {provincia}: {len(municipios_urls)}')

2026-04-21 15:54:42.775 | INFO     | tfm_lib.idealista_scrap.scraper:scrape_provinces:35 - Scrapeando provincia: https://www.idealista.com/venta-viviendas/murcia-provincia/municipios
2026-04-21 15:55:09.687 | DEBUG    | tfm_lib.idealista_scrap.scraper:scrape_provinces:37 - Response status: 200
2026-04-21 15:55:09.734 | INFO     | tfm_lib.idealista_scrap.parser:parse_province_urls:75 - Municipios encontrados: 46


Municipios encontrados en murcia: 46


## Transformación y guardado de datos
Para la provincia indicada se extraen la lista de urls de los municipios de dicha provincia. Añadimos la provincia y el municipio para guardar la información.

In [4]:
spark = tfm_utils.get_spark_session(f'Idealista_Municipios_{provincia}')

# scrape_provinces ahora devuelve List[Dict] con 'municipio_url' y 'num_ofertas'
df_municipios = (spark.createDataFrame(municipios_urls)
    .withColumn("provincia", F.lit(provincia))
    .withColumn("municipio_raw", F.regexp_extract(F.col("municipio_url"), r"/([^/]+)/?$", 1))
    .withColumn("municipio", F.regexp_replace(F.col("municipio_raw"), rf"-{provincia}$", ""))
    .withColumn("total_paginas", F.greatest(F.ceil(F.col("num_ofertas") / 30), F.lit(1)))
    .withColumn("_scraped", F.lit(0))
    .drop("municipio_raw")
)

tfm_utils.display(df_municipios)

,municipio_url,num_ofertas,provincia,municipio,total_paginas,_scraped
0,https://www.idealista.com/venta-viviendas/aban...,148,murcia,abanilla,5,0
1,https://www.idealista.com/venta-viviendas/abar...,99,murcia,abaran,4,0
2,https://www.idealista.com/venta-viviendas/agui...,921,murcia,aguilas,31,0
3,https://www.idealista.com/venta-viviendas/albu...,8,murcia,albudeite,1,0
4,https://www.idealista.com/venta-viviendas/alca...,227,murcia,alcantarilla,8,0
5,https://www.idealista.com/venta-viviendas/aled...,22,murcia,aledo,1,0
6,https://www.idealista.com/venta-viviendas/algu...,137,murcia,alguazas,5,0
7,https://www.idealista.com/venta-viviendas/alha...,131,murcia,alhama-de-murcia,5,0
8,https://www.idealista.com/venta-viviendas/arch...,160,murcia,archena,6,0
9,https://www.idealista.com/venta-viviendas/beni...,31,murcia,beniel,2,0


In [ ]:
municipios_table="raw.idealista.municipios_urls"

municipios_path=tfm_utils.full_table_path(municipios_table)

#Sobrescribimos por provincia y municipio
(df_municipios
    .write
    .partitionBy("provincia","municipio")
    .option("mergeSchema","true")
    .mode("overwrite")
    .format("delta")
    .save(municipios_path)
)